In [1]:
!pip install psutil

In [3]:
import time
import numpy as np
import psutil

proces = psutil.Process()

def snapshot():
    """Zwraca aktualne zużycie CPU i RAM."""
    mem = psutil.virtual_memory()
    return {
        "cpu_system_%": psutil.cpu_percent(interval=1),      # CPU całego systemu (próbkowanie 1 s)
        "ram_system_%": mem.percent,                          # RAM całego systemu
        "ram_proces_MB": proces.memory_info().rss / 1024**2,  # RAM tego procesu (kernela Jupytera)
    }

def raport(przed, po, czas):
    print(f"{'Metryka':<18}{'Przed':>12}{'Po':>12}{'Różnica':>12}")
    for k in przed:
        print(f"{k:<18}{przed[k]:>12.2f}{po[k]:>12.2f}{po[k] - przed[k]:>12.2f}")
    print(f"\nCzas wykonania: {czas:.2f} s")

# --- Pomiar przed ---
przed = snapshot()

# --- Obliczenia (przykład: duża macierz) ---
psutil.cpu_percent(interval=None)   # reset licznika CPU
start = time.perf_counter()

rng = np.random.default_rng(42)
macierz = rng.random((3000, 3000))
wynik = macierz @ macierz            # mnożenie macierzy

czas = time.perf_counter() - start
cpu_w_trakcie = psutil.cpu_percent(interval=None)  # średnie CPU w trakcie obliczeń

# --- Pomiar po ---
po = snapshot()

raport(przed, po, czas)
print(f"Średnie CPU systemu w trakcie obliczeń: {cpu_w_trakcie:.1f}%")

Metryka                  Przed          Po     Różnica
cpu_system_%             13.90        9.90       -4.00
ram_system_%             89.10       89.90        0.80
ram_proces_MB            88.70      234.14      145.44

Czas wykonania: 0.47 s
Średnie CPU systemu w trakcie obliczeń: 82.6%


In [4]:
import time
import numpy as np

rng = np.random.default_rng(42)
A = rng.random((1000, 1000))

start = time.perf_counter()
A_inv = np.linalg.inv(A)
czas = time.perf_counter() - start

print("Kształt odwrotności:", A_inv.shape)
print(f"Czas obliczeń: {czas:.3f} s")
print(f"Uwarunkowanie macierzy: {np.linalg.cond(A):.3e}")
print(f"Maks. błąd |A @ A_inv - I|: {np.abs(A @ A_inv - np.eye(1000)).max():.3e}")
print("Poprawna odwrotność:", np.allclose(A @ A_inv, np.eye(1000)))

Kształt odwrotności: (1000, 1000)
Czas obliczeń: 0.144 s
Uwarunkowanie macierzy: 2.123e+05
Maks. błąd |A @ A_inv - I|: 6.981e-13
Poprawna odwrotność: True


In [5]:
import time
import numpy as np
import psutil

proces = psutil.Process()

def snapshot():
    mem = psutil.virtual_memory()
    return {
        "cpu_system_%": psutil.cpu_percent(interval=1),
        "ram_system_%": mem.percent,
        "ram_system_uzyty_MB": mem.used / 1024**2,
        "ram_proces_MB": proces.memory_info().rss / 1024**2,
    }

przed = snapshot()
psutil.cpu_percent(interval=None)   # reset licznika CPU
start = time.perf_counter()

rng = np.random.default_rng(42)
A = rng.random((1000, 1000))
A_inv = np.linalg.inv(A)

czas = time.perf_counter() - start
cpu_w_trakcie = psutil.cpu_percent(interval=None)
po = snapshot()

print(f"{'Metryka':<22}{'Przed':>12}{'Po':>12}{'Różnica':>12}")
for k in przed:
    print(f"{k:<22}{przed[k]:>12.2f}{po[k]:>12.2f}{po[k]-przed[k]:>12.2f}")
print(f"\nCzas obliczeń: {czas:.3f} s")
print(f"Średnie CPU w trakcie: {cpu_w_trakcie:.1f}%")
print(f"Rozmiar A: {A.nbytes/1024**2:.2f} MB, A_inv: {A_inv.nbytes/1024**2:.2f} MB")
print("Rdzenie logiczne:", psutil.cpu_count())

Metryka                      Przed          Po     Różnica
cpu_system_%                 33.90       17.20      -16.70
ram_system_%                 91.00       91.00        0.00
ram_system_uzyty_MB       14706.14    14710.21        4.07
ram_proces_MB               251.94      251.94        0.00

Czas obliczeń: 0.054 s
Średnie CPU w trakcie: 63.0%
Rozmiar A: 7.63 MB, A_inv: 7.63 MB
Rdzenie logiczne: 8
